# 单个 ETF 过去 N 个月收益率

本 notebook 基于 ETF 动量轮动数据，计算每个 ETF 在过去 `1/3/6/12/24/36/48/60/72/84/96/108/120` 个月的收益率。

用途：

- 作为 ETF 板块动量轮动的基础信号；
- 后续可以按某个 N 月窗口做横截面排名；
- 也可以比较不同回看窗口的稳定性。

价格口径：东方财富前复权日线，项目记录为 `adjust=qfq`。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.tseries.offsets import DateOffset

BASE = Path.cwd()
if BASE.name.lower() in {"notesbooks", "notebooks"}:
    BASE = BASE.parent

DATA_PATH = BASE / "data" / "etf_momentum_daily_eastmoney_qfq.csv"
OUTPUT_PATH = BASE / "outputs" / "etf_single_momentum_returns.csv"
REPORT_PATH = BASE / "outputs" / "etf_single_momentum_returns_report.md"

MONTH_WINDOWS = [1, 3, 6, 12, 24, 36, 48, 60, 72, 84, 96, 108, 120]

DATA_PATH, OUTPUT_PATH, REPORT_PATH


## 计算口径

收益率公式：

```text
return_n_months = current_close / lookback_close - 1
```

回看日期口径：

1. 当前交易日向前推 N 个自然月；
2. 如果目标日期不是交易日，取该 ETF 在目标日期之前最近一个可用交易日；
3. 如果没有足够历史数据，则 `valid=False`。


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"], dtype={"symbol": "string"})
required = {"date", "symbol", "name", "bucket", "theme", "close", "adjust", "source"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("Input data is empty")
bad_adjust = sorted(set(df["adjust"].dropna()) - {"qfq"})
if bad_adjust:
    raise ValueError(f"Unexpected adjust values: {bad_adjust}")

print(f"rows={len(df):,}, etfs={df['symbol'].nunique()}, start={df['date'].min().date()}, end={df['date'].max().date()}")
df.head()


In [ ]:
def compute_single_symbol_returns(symbol_df: pd.DataFrame, months_list: list[int]) -> pd.DataFrame:
    symbol_df = symbol_df.sort_values("date").reset_index(drop=True).copy()
    symbol_df["date"] = pd.to_datetime(symbol_df["date"])

    dates = symbol_df["date"].to_numpy(dtype="datetime64[ns]")
    closes = symbol_df["close"].astype(float).to_numpy()
    n_rows = len(symbol_df)
    parts = []

    for months in months_list:
        target_dates = symbol_df["date"] - DateOffset(months=months)
        target_np = target_dates.to_numpy(dtype="datetime64[ns]")
        lookback_idx = np.searchsorted(dates, target_np, side="right") - 1

        has_past = lookback_idx >= 0
        lookback_dates = np.full(n_rows, np.datetime64("NaT"), dtype="datetime64[ns]")
        lookback_closes = np.full(n_rows, np.nan)

        lookback_dates[has_past] = dates[lookback_idx[has_past]]
        lookback_closes[has_past] = closes[lookback_idx[has_past]]

        valid = has_past & np.isfinite(lookback_closes) & (lookback_closes > 0)
        returns = np.full(n_rows, np.nan)
        returns[valid] = closes[valid] / lookback_closes[valid] - 1

        actual_days = np.full(n_rows, np.nan)
        actual_days[valid] = (dates[valid] - lookback_dates[valid]).astype("timedelta64[D]").astype(float)

        parts.append(
            pd.DataFrame(
                {
                    "date": symbol_df["date"],
                    "symbol": symbol_df["symbol"],
                    "name": symbol_df["name"],
                    "bucket": symbol_df["bucket"],
                    "theme": symbol_df["theme"],
                    "months": months,
                    "lookback_target_date": target_dates,
                    "lookback_actual_date": pd.to_datetime(lookback_dates),
                    "actual_calendar_days": actual_days,
                    "current_close": closes,
                    "lookback_close": lookback_closes,
                    "return_n_months": returns,
                    "valid": valid,
                    "adjust": symbol_df["adjust"],
                    "source": symbol_df["source"],
                }
            )
        )

    return pd.concat(parts, ignore_index=True).sort_values(["date", "months"]).reset_index(drop=True)


In [ ]:
result = pd.concat(
    [
        compute_single_symbol_returns(symbol_df, MONTH_WINDOWS)
        for _, symbol_df in df.sort_values(["symbol", "date"]).groupby("symbol", sort=True)
    ],
    ignore_index=True,
)
print(f"result rows={len(result):,}, etfs={result['symbol'].nunique()}, windows={MONTH_WINDOWS}")
result.groupby(["months", "valid"]).size()


In [ ]:
latest_date = result["date"].max()
latest = result[(result["date"] == latest_date) & (result["valid"])].copy()
latest_rank = latest.sort_values(["months", "return_n_months"], ascending=[True, False])

print(f"latest date: {latest_date.date()}")
latest_rank[latest_rank["months"].isin([1, 3, 6, 12])][
    ["months", "symbol", "name", "bucket", "theme", "lookback_actual_date", "return_n_months"]
].head(40)


In [ ]:
# ??????24/36/48/60/72??????????
latest_rank[latest_rank["months"].isin([24, 36, 48, 60, 72])][
    ["months", "symbol", "name", "bucket", "theme", "lookback_actual_date", "return_n_months"]
].head(80)


In [ ]:
# Extra-long windows: 84/96/108/120 month latest ranking
latest_rank[latest_rank["months"].isin([84, 96, 108, 120])][
    ["months", "symbol", "name", "bucket", "theme", "lookback_actual_date", "return_n_months"]
].head(80)


In [ ]:
def pct(value) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.2%}"


def make_report(result: pd.DataFrame, months_list: list[int], output_path: Path) -> str:
    latest_date = result["date"].max()
    latest = result[(result["date"] == latest_date) & result["valid"]].copy()
    latest = latest.sort_values(["months", "return_n_months"], ascending=[True, False])
    coverage = (
        result.groupby(["symbol", "name", "bucket", "theme", "months"], dropna=False)
        .agg(rows=("date", "size"), valid_rows=("valid", "sum"), start=("date", "min"), end=("date", "max"))
        .reset_index()
    )

    lines = [
        "# 单个 ETF 过去 N 个月收益率",
        "",
        "## 输出文件",
        "",
        f"- 明细结果：`{output_path.as_posix()}`",
        "",
        "## 计算口径",
        "",
        "- 输入价格：`data/etf_momentum_daily_eastmoney_qfq.csv`。",
        "- 价格口径：`adjust=qfq`，前复权收盘价 `close`。",
        "- 收益率公式：`return_n_months = current_close / lookback_close - 1`。",
        "- 回看日期：从当前交易日向前推 N 个自然月；若目标日期不是交易日，取该 ETF 在目标日期之前最近一个可用交易日。",
        "- 输出形态：长表；一行代表某 ETF 在某交易日、某 N 月窗口下的过去收益率。",
        "",
        "## 本次参数",
        "",
        f"- N 月窗口：{', '.join(map(str, months_list))}",
        f"- 最新日期：{latest_date.date()}",
        f"- 结果行数：{len(result)}",
        "",
        "## 最新日期截面的动量收益率",
        "",
        "| N个月 | 分层 | 代码 | 名称 | 主题 | 当前收盘 | 回看日期 | 回看收盘 | 过去N月收益率 |",
        "|---:|---|---|---|---|---:|---|---:|---:|",
    ]
    for row in latest.itertuples(index=False):
        lines.append(
            f"| {row.months} | {row.bucket} | {row.symbol} | {row.name} | {row.theme} | "
            f"{row.current_close:.4f} | {pd.Timestamp(row.lookback_actual_date).date()} | "
            f"{float(row.lookback_close):.4f} | {pct(row.return_n_months)} |"
        )

    lines.extend([
        "",
        "## 有效样本覆盖",
        "",
        "| 代码 | 名称 | 分层 | 主题 | N个月 | 总行数 | 有效行数 | 起始 | 结束 |",
        "|---|---|---|---|---:|---:|---:|---|---|",
    ])
    for row in coverage.sort_values(["bucket", "symbol", "months"]).itertuples(index=False):
        lines.append(
            f"| {row.symbol} | {row.name} | {row.bucket} | {row.theme} | {row.months} | "
            f"{row.rows} | {int(row.valid_rows)} | {pd.Timestamp(row.start).date()} | {pd.Timestamp(row.end).date()} |"
        )
    return "\n".join(lines) + "\n"


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

result.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
REPORT_PATH.write_text(make_report(result, MONTH_WINDOWS, OUTPUT_PATH), encoding="utf-8")

print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {REPORT_PATH}")
